# Credit News Factors: Screen, Drill, Discover Narrative

A credit-analyst workflow built on three [Bigdata.com](https://bigdata.com) MCP tools,
orchestrated with an LLM (`gpt-5.6-terra`):

| Step | Tool | What it does |
|---|---|---|
| 1. **Rank the universe** | `bigdata_screen_credit_factor` | Runs a negative screen across a portfolio/sector list and returns the names trending worst on credit-news sentiment. |
| 2. **Drill into a name** | `bigdata_get_credit_factor` | Pulls one deteriorating name's most extreme catalyst rows, by event type. |
| 3. **Build the narrative** | Your LLM + `bigdata_search` | Hands the catalyst rows to the LLM with supporting news search, which explains *why* the score moved and *what to watch next*. |

> The score tells you something moved. The catalyst rows tell you what.

**Demo universe:** mega-cap tech plus a few staples (Apple, Microsoft, Alphabet, Amazon, Meta, Nvidia, Tesla, Walmart, and others) — swap in
your own portfolio or sector coverage list to reuse this notebook as-is.

All MCP session handling and prompt construction is kept out of this notebook and lives in
[`src/bigdata_mcp_client.py`](src/bigdata_mcp_client.py) and [`src/narrative.py`](src/narrative.py) —
this notebook only calls those helpers and inspects the results.


## Prerequisites

- **`BIGDATA_API_KEY`** — a [Bigdata.com](https://bigdata.com) API key (Developer Platform → API Keys), used to
  connect to the Bigdata.com Remote MCP server at `https://mcp.bigdata.com/`.
- **`OPENAI_API_KEY`** — used to call the `gpt-5.6-terra` model for narrative synthesis in Step 3.
- Copy `.env.example` to `.env` and fill in both keys.
- Dependencies installed from `requirements.txt`:

```bash
uv venv
source .venv/bin/activate
uv pip install -r requirements.txt
```

Both keys are read from `.env` — never hardcode a key in the notebook.


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

script_dir = Path.cwd().resolve()
load_dotenv(script_dir / ".env")

src_dir = str(script_dir / "src")
if src_dir not in sys.path:
    sys.path.append(src_dir)

from bigdata_mcp_client import BigdataClient
from narrative import build_narrative

assert os.environ.get("BIGDATA_API_KEY"), "Set BIGDATA_API_KEY in .env before running this notebook."
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env before running this notebook."

pd.set_option("display.max_colwidth", 120)
print("Environment OK — Bigdata.com and OpenAI keys are set.")


Environment OK — Bigdata.com and OpenAI keys are set.


## Define the universe

Resolve each company name to its RavenPack entity ID via `find_securities` — the ID other
MCP tools take as input. Swap `MAG7` for any list of company names to point this notebook at
a different portfolio or sector.


In [2]:
MAG7 = [
    "Apple",
    "Microsoft",
    "Alphabet",
    "Amazon",
    "Meta Platforms",
    "Nvidia",
    "Tesla",
    "Walmart",
    "Coca-Cola",
    "Procter & Gamble",
    "Johnson & Johnson",
    "Merck & Co",
    "Pfizer",
    
]

async with BigdataClient.connect() as client:
    entity_ids = {name: await client.resolve_entity_id(name) for name in MAG7}

id_to_name = {v: k for k, v in entity_ids.items()}
pd.DataFrame(entity_ids.items(), columns=["company", "rp_entity_id"])


,company,rp_entity_id
0,Apple,D8442A
1,Microsoft,228D42
2,Alphabet,4A6F00
3,Amazon,0157B1
4,Meta Platforms,12E454
5,Nvidia,E09E2B
6,Tesla,DD3BB1
7,Walmart,713810
8,Coca-Cola,EEA6B3
9,Procter & Gamble,2E61CC


## Step 1 — `screen_credit_factor`: rank the universe

Run a **negative** screen across the coverage universe on a **weekly**-smoothed horizon (7-day decay,
smooths out single-day noise). Companies are ranked by their single most extreme negative
credit-news catalyst — worst first.


In [3]:
HORIZON = "weekly"

async with BigdataClient.connect() as client:
    screen_df = await client.screen_credit_factor(
        entity_universe=list(entity_ids.values()),
        horizon=HORIZON,
        screen_direction="negative",
        company_limit=len(MAG7),
        scores_per_company=3,
    )

worst_catalyst_per_company = (
    screen_df.sort_values(["company_rank", "factor_rank"])
    .groupby("entity_name", sort=False)
    .first()[["company_rank", "credit_event_group", "credit_event_type", "credit_factor_score"]]
    .rename(columns={
        "credit_event_group": "worst_event_group",
        "credit_event_type": "worst_event_type",
        "credit_factor_score": "worst_score",
    })
)
worst_catalyst_per_company


,company_rank,worst_event_group,worst_event_type,worst_score
entity_name,,,,
Walmart Inc.,1,price-targets,price-target,-0.197
Tesla Inc.,2,products-services,product-recall,-0.115
Procter & Gamble Co.,3,insider-trading,insider-sell,-0.095
NVIDIA Corp.,4,earnings,earnings,-0.081
Amazon.com Inc.,5,insider-trading,insider-sell,-0.051
Apple Inc.,6,products-services,product-pricing,-0.042
Coca-Cola Co.,7,insider-trading,insider-sell,-0.041
Meta Platforms Inc.,8,insider-trading,insider-sell,-0.028
Merck & Co. Inc.,9,insider-trading,insider-sell,-0.017


**Reading this table:** `worst_score` is the `MEAN_CREDIT_SENTIMENT` factor
(range −1 to 1; higher = lower credit risk), so the most negative score at the top is the
name trending worst on credit-news sentiment this week. That name is our candidate for
Step 2 — "pick a deteriorating name."


In [4]:
id_to_name = {v: k for k, v in entity_ids.items()}
top_row = screen_df.sort_values(["company_rank", "factor_rank"]).iloc[0]
target_id, target_name = top_row["rp_entity_id"], id_to_name[top_row["rp_entity_id"]]

print(f"Deteriorating name flagged by the screen: {target_name} ({target_id})")
print(f"Worst catalyst: {top_row['credit_event_group']} / {top_row['credit_event_type']} "
      f"(score {top_row['credit_factor_score']:+.3f})")


Deteriorating name flagged by the screen: Walmart (713810)
Worst catalyst: price-targets / price-target (score -0.197)


## Step 2 — `get_credit_factor`: drill into the flagged name

Pull the deteriorating name's most extreme **negative** *and* **positive** catalyst rows on
the same horizon, so we see the full balance of what's driving the score — not just the
single worst headline.


In [5]:
print(f"Drilling into: {target_name} ({target_id})")

async with BigdataClient.connect() as client:
    catalyst_df = await client.get_credit_factor(
        rp_entity_id=target_id,
        horizon=HORIZON,
        negative_limit=5,
        positive_limit=3,
    )

display_cols = [
    "catalyst_direction", "catalyst_rank", "credit_event_group",
    "credit_event_type", "credit_factor_score", "timestamp_utc",
]
catalyst_df[display_cols].sort_values(
    ["catalyst_direction", "catalyst_rank"], ascending=[False, True]
)


Drilling into: Walmart (713810)


,catalyst_direction,catalyst_rank,credit_event_group,credit_event_type,credit_factor_score,timestamp_utc
5,positive,1,products-services,product-release,0.078,2026-08-26T20:00:00
6,positive,2,revenues,revenue,0.066,2026-08-26T20:00:00
7,positive,3,earnings,earnings-guidance,0.038,2026-08-26T20:00:00
0,negative,1,price-targets,price-target,-0.197,2026-08-26T20:00:00
1,negative,2,earnings,earnings-estimate,-0.035,2026-08-26T20:00:00
2,negative,3,stock-prices,stock-price,-0.030,2026-08-26T20:00:00
3,negative,4,earnings,earnings-per-share-guidance-expectations,-0.016,2026-08-26T20:00:00
4,negative,5,revenues,same-store-sales,-0.015,2026-08-26T20:00:00


**Callout:** the score told us this name was trending worst. This table tells us
*what* — the specific credit-news event groups and types ranked by how much each moved the
factor. The top-ranked negative row is what we'll dig into with news search next.


## Step 3 — Your LLM + `bigdata_search`: build the narrative

For the top negative catalyst rows, pull supporting news evidence with `bigdata_search`,
then hand the catalyst data + evidence to `gpt-5.6-terra`, which explains *why* the score
moved and *what to watch next*.


In [6]:
top_negative = (
    catalyst_df[catalyst_df["catalyst_direction"] == "negative"]
    .sort_values("catalyst_rank")
    .head(2)
)

evidence_by_catalyst = {}
async with BigdataClient.connect() as client:
    for _, row in top_negative.iterrows():
        query = f"{target_name} {row['credit_event_type'].replace('-', ' ')}"
        evidence_by_catalyst[row["credit_event_type"]] = await client.search_news(
            query, context="recent news, last few weeks", max_chunks=6
        )

{catalyst_type: len(docs) for catalyst_type, docs in evidence_by_catalyst.items()}


{'price-target': 3, 'earnings-estimate': 6}

In [7]:
for catalyst_type, docs in evidence_by_catalyst.items():
    print(f"\n=== {catalyst_type} — top evidence ===")
    for d in docs[:3]:
        print(f"- {d['headline']}  ({d['source']['name']}, {d['timestamp'][:10]})")



=== price-target — top evidence ===
- Daiwa Securities Adjusts Price Target on Walmart to $113 From $132  (MT Newswires, 2026-08-26)
- Walmart's Big Sell-Off Could Be a Buying Opportunity  (AOL.com, 2026-08-27)
- Walmart price target lowered to $126 from $145 at BMO Capital  (The Fly, 2026-08-21)

=== earnings-estimate — top evidence ===
- Walmart Stock Moves Lower Thursday: What's Going On?  (Benzinga, 2026-08-27)
- Walmart Just Dropped 10% in a Month. Is It Time to Sell?  (AOL.com, 2026-08-27)
- Oppenheimer sees bottom for Walmart in low $90s to low $100s  (The Fly, 2026-08-25)


In [8]:
from openai import OpenAI

llm_client = OpenAI()  # reads OPENAI_API_KEY from the environment

narrative = build_narrative(
    entity_name=target_name,
    horizon=HORIZON,
    catalyst_df=catalyst_df,
    evidence_by_catalyst=evidence_by_catalyst,
    client=llm_client,
)
display(Markdown(narrative))


### What moved  
Walmart’s weekly credit-news sentiment turned negative mainly because market commentary focused on a post-earnings valuation reset and softer U.S. sales momentum. The largest negative catalyst was price-target cuts: Daiwa reduced its target to **$113 from $132** on August 26, while BMO had cut its target to **$126 from $145** on August 21, citing a greater-than-expected deceleration in Q2 U.S. comparable sales, including Health & Wellness ticket headwinds. Earnings-related coverage also emphasized that Walmart’s Q2 result, despite beating expectations and raising guidance, was overshadowed by its **slowest U.S. comparable-sales growth in six years** and margin concerns. The share-price reaction reinforced this narrative: commentary cited a roughly **12% decline after the August 20 fiscal Q2 2027 release**. Positive news—raised full-year sales, operating-income and EPS guidance, plus strong marketplace, advertising and membership growth—was not enough to offset the negative market reassessment.

### Why it matters for credit  
The news is not pointing to an immediate balance-sheet or refinancing problem; rather, it raises a watch item around the durability of operating cash flow and margins. Slower U.S. comp growth, particularly in categories affected by lower ticket sizes, could constrain earnings growth if it persists, while investor commentary specifically flagged pressure on core-retail margins. That matters because Walmart’s cash generation ultimately depends on maintaining profitable store sales as it invests in e-commerce and other growth initiatives. Offsetting this concern, management raised fiscal-year net-sales guidance to **4.0%–5.0% from 3.5%–4.5%** and adjusted EPS guidance to **$2.80–$2.87**; marketplace sales grew **52%**, advertising revenue **38%**, and membership revenue **17%**. Credit focus should therefore be whether these higher-margin or fee-based businesses can offset any further weakening in core U.S. retail sales and margins.

### What to watch next  
- **Next quarterly earnings release and 10-Q:** Track U.S. comparable-sales growth, especially Health & Wellness, and management’s explanation of ticket and traffic trends.  
- **Guidance maintenance:** Monitor whether Walmart reiterates the fiscal-year **4.0%–5.0% net-sales** and **$2.80–$2.87 adjusted EPS** outlook, or cites additional pressure on operating income/margins.  
- **Cash-flow and liquidity disclosures:** Review the next filing for operating cash flow, capital expenditures, free-cash-flow trends, debt maturities and any change in borrowing or share-repurchase activity.  
- **Analyst estimate and target revisions:** Watch for further cuts following the August reductions by Daiwa (**$132 to $113**) and BMO (**$145 to $126**), particularly if tied to reduced earnings estimates rather than valuation multiples.

## Recap

1. **Screen** (`bigdata_screen_credit_factor`) ranked the coverage universe and surfaced the
   worst-trending name on credit-news sentiment.
2. **Drill** (`bigdata_get_credit_factor`) broke that name's score down into the specific
   catalyst rows — event group, type, and how much each moved the score.
3. **Discover narrative** (`bigdata_search` + `gpt-5.6-terra`) turned those catalyst rows
   into a grounded explanation of what moved, why it matters for credit, and what to watch
   next.

**Reuse this notebook** by swapping `MAG7` for your own portfolio or sector coverage list,
changing `HORIZON` to `"daily"` or `"monthly"`, or flipping `screen_direction` to
`"positive"` to find improving credit stories instead.

Powered by [Bigdata.com](https://bigdata.com).
